In [1]:
from molpipeline import Pipeline
from molpipeline.any2mol import AutoToMol
from molpipeline.mol2any import MolToMorganFP
from molpipeline.mol2mol import (
    ElementFilter,
    SaltRemover,
)

In [2]:
import pandas as pd

smiles = pd.read_csv('compounds_with_smiles.csv', sep = '\t')

In [3]:
pipeline = Pipeline([
      ("auto2mol", AutoToMol()),                                     # reading molecules
      ("element_filter", ElementFilter()),                           # standardization
      ("salt_remover", SaltRemover()),                               # standardization
    ],
    )

In [4]:
mols = pipeline.transform(smiles['smiles'].values)

In [5]:
from rdkit import Chem
from molpipeline.error_handling import InvalidInstance


n_invalid = 0
n_changed = 0
n_unchanged = 0

for smi_in, mol in zip(smiles["smiles"], mols):

    if isinstance(mol, InvalidInstance):
        n_invalid += 1
        continue

    smi_out = Chem.MolToSmiles(mol, canonical=True)

    # canonicalize original SMILES as well
    mol_in = Chem.MolFromSmiles(smi_in)

    if mol_in is None:
        continue

    smi_in_canonical = Chem.MolToSmiles(mol_in, canonical=True)

    if smi_in_canonical != smi_out:
        n_changed += 1
    else:
        n_unchanged += 1

print(f"Changed: {n_changed}")
print(f"Unchanged: {n_unchanged}")
print(f"Invalid: {n_invalid}")

[09:13:50] WARNING: not removing hydrogen atom without neighbors


Changed: 1088
Unchanged: 7079
Invalid: 618


In [6]:
from rdkit import Chem
from molpipeline.error_handling import InvalidInstance

clean_smiles = []

for mol in mols:
    if isinstance(mol, InvalidInstance):
        clean_smiles.append(None)
    else:
        clean_smiles.append(Chem.MolToSmiles(mol))

smiles["clean_smiles"] = clean_smiles

In [ ]:
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import Descriptors

# Common metals for screening
METALS = {
    "Li", "Na", "K", "Rb", "Cs",
    "Be", "Mg", "Ca", "Sr", "Ba",
    "Sc", "Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn",
    "Y", "Zr", "Nb", "Mo", "Tc", "Ru", "Rh", "Pd", "Ag", "Cd",
    "Hf", "Ta", "W", "Re", "Os", "Ir", "Pt", "Au", "Hg",
    "Al", "Ga", "In", "Tl",
    "Ge", "Sn", "Pb",
    "La", "Ce", "Pr", "Nd", "Pm", "Sm", "Eu", "Gd", "Tb",
    "Dy", "Ho", "Er", "Tm", "Yb", "Lu",
    "Ac", "Th", "Pa", "U", "Np", "Pu"
}


def classify_molecule(smiles: str) -> str:
    """
    Heuristic classification into:
      - ordinary organic
      - charged organic
      - salt
      - metal-containing
      - metal complex
      - very small (< 5 atoms)
      - mixture
    """

    if pd.isna(smiles):
        return "invalid"

    smiles = str(smiles).strip()

    try:
        mol = Chem.MolFromSmiles(smiles)
    except Exception:
        return "invalid"

    if mol is None:
        return "invalid"

    fragments = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=False)
    # Mixtures / disconnected systems
    if len(fragments) > 1:
        
        metal_fragments = 0
        charged_fragments = 0

        for frag in fragments:
            has_metal = any(
                atom.GetSymbol() in METALS
                for atom in frag.GetAtoms()
            )
            if has_metal:
                metal_fragments += 1

            charge = sum(atom.GetFormalCharge() for atom in frag.GetAtoms())
            if charge != 0:
                charged_fragments += 1

        if metal_fragments > 0:
            return "metal complex"

        if charged_fragments >= 2:
            return "salt"

        return "mixture"

    atoms = mol.GetNumAtoms()

    if atoms < 5:
        return "very small (< 5 atoms)"

    has_metal = any(
        atom.GetSymbol() in METALS
        for atom in mol.GetAtoms()
    )

    if has_metal:
        return "metal-containing"

    total_charge = sum(
        atom.GetFormalCharge()
        for atom in mol.GetAtoms()
    )

    if total_charge != 0:
        return "charged organic"

    return "ordinary organic"


def has_nan_or_inf_rdkit_descriptors(mol) -> bool:
    """
    Returns True if any RDKit descriptor is NaN or Inf.
    """
    for name, func in Descriptors.descList:
        try:
            value = func(mol)

            if value is None:
                return True

            if isinstance(value, (float, np.floating)):
                if np.isnan(value) or np.isinf(value):
                    return True

        except Exception:
            return True

    return False


def create_smiles_class_summary(
    df: pd.DataFrame,
    smiles_column: str,
    output_tsv: str
) -> pd.DataFrame:
    """
    Creates a TSV summary containing:

        class
        count
        % aller Moleküle
        % mit NaN/Inf in RDKit physchem descriptors

    Parameters
    ----------
    df : pd.DataFrame
    smiles_column : str
        Column containing SMILES.
    output_tsv : str
        Path to output TSV.

    Returns
    -------
    pd.DataFrame
        Summary table.
    """

    records = []

    for smiles in df[smiles_column].values:
        mol = Chem.MolFromSmiles(str(smiles)) if pd.notna(smiles) else None

        mol_class = classify_molecule(smiles)

        has_bad_desc = False
        if mol is not None:
            has_bad_desc = has_nan_or_inf_rdkit_descriptors(mol)

        records.append({
            "class": mol_class,
            "bad_desc": has_bad_desc
        })

    tmp = pd.DataFrame(records)

    total_molecules = len(df)

    summary_rows = []

    for cls, grp in tmp.groupby("class"):
        count = len(grp)

        pct_all = 100.0 * (count / total_molecules)

        pct_bad = (
            100.0 * grp["bad_desc"].sum() / count
            if count > 0 else 0.0
        )

        summary_rows.append({
            "catgory": cls,
            "count": count,
            "% molecules": round(pct_all, 2),
            "% with NaN/Inf in RDKit physchem descriptors": round(pct_bad, 2)
        })

    summary = (
        pd.DataFrame(summary_rows)
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )

    summary.to_csv(
        output_tsv,
        sep="\t",
        index=False
    )

    return summary

In [68]:
create_smiles_class_summary(smiles, smiles_column='smiles', output_tsv='summary_table.csv')

,class,count,% aller Moleküle,% mit NaN/Inf in RDKit physchem descriptors
0,ordinary organic,6866,78.16,0.33
1,mixture,919,10.46,5.01
2,metal complex,489,5.57,99.80
3,salt,374,4.26,7.22
4,very small (< 5 atoms),82,0.93,14.63
5,metal-containing,50,0.57,88.00
6,charged organic,4,0.05,0.00
7,invalid,1,0.01,0.00


In [18]:
# remove invalid molecules
df_clean = smiles.dropna(subset=["clean_smiles"]).copy()

# remove duplicates based on cleaned smiles
n_before = len(df_clean)
print(df_clean.loc[df_clean['compound'] == 'DTXSID5023825', :])
df_clean = df_clean.drop_duplicates(subset="clean_smiles", keep='first')

n_after = len(df_clean)

print(f"Removed {n_before - n_after} duplicates")
print(f"Remaining molecules: {n_after}")


Empty DataFrame
Columns: [compound, smiles, inchi, inchikey, clean_smiles]
Index: []
Removed 207 duplicates
Remaining molecules: 7960


In [19]:
df_clean.drop(columns='smiles', inplace=True)
df_clean.rename(columns={'clean_smiles':'smiles'}, inplace=True)

In [20]:
df_clean.to_csv('clean_smiles.csv', sep = '\t', index=False)

In [26]:
df_clean.dropna(subset=['compound', 'smiles'])

,compound,inchi,inchikey,smiles
0,DTXSID9046499,InChI=1S/C20H26ClNO5.ClH/c1-5-22(6-2)11-10-15-...,CCCZJRFQJNGCCU-UHFFFAOYSA-N,CCOC(=O)COc1ccc2c(C)c(CCN(CC)CC)c(=O)oc2c1Cl
1,DTXSID3049386,InChI=1S/C9H20N.F6P/c1-3-7-10(2)8-5-4-6-9-10;1...,CDBFWKDOZHJELY-UHFFFAOYSA-N,CCC[N+]1(C)CCCCC1
3,DTXSID3045568,InChI=1S/C20H22ClN3O.2ClH.2H2O/c1-3-24(4-2)13-...,YVNAYSHNIILOJS-UHFFFAOYSA-N,CCN(CC)Cc1cc(Nc2ccnc3cc(Cl)ccc23)ccc1O
4,DTXSID6026298,"InChI=1S/C8H10/c1-7-4-3-5-8(2)6-7/h3-6H,1-2H3",IVSZLXZYQVIEFR-UHFFFAOYSA-N,Cc1cccc(C)c1
5,DTXSID3047847,InChI=1S/C19H21NO3.ClH/c1-2-8-20-9-7-19-12-4-6...,NAHATSPWSULUAA-ZQGPYOJVSA-N,C=CCN1CC[C@]23c4c5ccc(O)c4O[C@H]2[C@@H](O)C=C[...
...,...,...,...,...
8779,DTXSID0057835,InChI=1S/C21H23N3O7S.ClH/c1-10-12(31-20(28)30-...,FXXSETTYJSGMCR-GLCLSGQWSA-N,Cc1oc(=O)oc1COC(=O)[C@@H]1N2C(=O)[C@@H](NC(=O)...
8780,DTXSID2044927,InChI=1S/C16H35O3P/c1-5-9-11-15(7-3)13-18-20(1...,HZIUHEQKVCPTAJ-UHFFFAOYSA-N,CCCCC(CC)CO[PH](=O)OCC(CC)CCCC
8781,DTXSID9045924,"InChI=1S/C29H30N6O6/c1-5-8-23-30-25(29(3,4)38)...",UQGKUQLKSCSZGY-UHFFFAOYSA-N,CCCc1nc(C(C)(C)O)c(C(=O)OCc2oc(=O)oc2C)n1Cc1cc...
8782,DTXSID7023146,InChI=1S/C6H12O6/c7-1-2(8)4(10)6(12)5(11)3(1)9...,CDAISMWEOUEBRE-GPIVLXJGSA-N,O[C@H]1[C@H](O)[C@@H](O)[C@H](O)[C@@H](O)[C@H]1O


In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import rdFingerprintGenerator

# Convert SMILES to molecules
mols = [Chem.MolFromSmiles(s) for s in df_clean["smiles"]]
compounds = df_clean['compound'].values
# -----------------------------
# 1. RDKit descriptors dataframe
# -----------------------------

descriptor_names = [name for name, _ in Descriptors._descList]

descriptor_rows = []
for i, mol in enumerate(mols):
    if mol is None:
        descriptor_rows.append([None] * (len(descriptor_names) + 1))
    else:
        try:
            descriptor_rows.append( [compounds[i]] + 
                [func(mol) for _, func in Descriptors._descList]
            )
        except:
            descriptor_rows.append([None] * (len(descriptor_names)+1))

desc_df = pd.DataFrame(
    descriptor_rows,
    columns=['compound'] + descriptor_names
)
desc_df.dropna(inplace=True)


# -----------------------------
# 2. Morgan fingerprint dataframe
# -----------------------------


morgan_gen = rdFingerprintGenerator.GetMorganGenerator()
n_bits = 2048
fp_rows = []
for i, mol in enumerate(mols):
    if mol is None:
        fp_rows.append([None] * (n_bits+1))
    else:
        fp = morgan_gen.GetFingerprint(mol)
        fp_rows.append([compounds[i]] + list(fp))

fp_cols = ['compound'] +[f"Morgan_{i}" for i in range(n_bits)]
fp_df = pd.DataFrame(
    fp_rows,
    columns=fp_cols
)

# Results
print(desc_df.shape)
print(fp_df.shape)


[09:32:13] WARNING: not removing hydrogen atom without neighbors
[09:32:31] WARNING: not removing hydrogen atom without neighbors
[09:32:47] 

****
Invariant Violation
Bond order must be Single, Double, Triple or Aromatic
Violation occurred on line 62 in file C:\rdkit\build\temp.win-amd64-cpython-313\Release\rdkit\Code\GraphMol\Descriptors\BCUT.cpp
Failed Expression: 0
****

[09:32:47] 

****
Invariant Violation
Bond order must be Single, Double, Triple or Aromatic
Violation occurred on line 62 in file C:\rdkit\build\temp.win-amd64-cpython-313\Release\rdkit\Code\GraphMol\Descriptors\BCUT.cpp
Failed Expression: 0
****

[09:32:47] 

****
Invariant Violation
Bond order must be Single, Double, Triple or Aromatic
Violation occurred on line 62 in file C:\rdkit\build\temp.win-amd64-cpython-313\Release\rdkit\Code\GraphMol\Descriptors\BCUT.cpp
Failed Expression: 0
****

[09:32:47] 

****
Invariant Violation
Bond order must be Single, Double, Triple or Aromatic
Violation occurred on line 62 in f

(7951, 218)
(7960, 2049)


In [31]:
desc_df.to_csv('cleaned_physchem.csv', sep = '\t', index=False)

In [34]:
fp_df.to_csv('cleaned_maccs.csv', sep = '\t', index = False)

In [33]:
from rdkit.Chem import MACCSkeys

n_bits = 167
fp_rows = []

for i, mol in enumerate(mols):
    if mol is None:
        fp_rows.append([compounds[i]] + [None] * n_bits)
    else:
        fp = MACCSkeys.GenMACCSKeys(mol)
        fp_rows.append([compounds[i]] + list(fp))

fp_cols = ['compound'] + [f"MACCS_{i}" for i in range(n_bits)]

fp_df = pd.DataFrame(
    fp_rows,
    columns=fp_cols
)